# Supervised classifier

Same data as `inspecting.ipynb`, but trained on both classes instead of accepted batches only.
A_101–A_104 are treated as rejected. Accepted = positive class (as on the leaderboard).

Predictions are written to `team-workspace/submissions/`, not the repo.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent / "src"))   # notebooks/ -> src/
from data_loader import INPUT_DIR, WORKSPACE, load_long, load_variables
from features import build_features, drop_constant
from models import pca_monitor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from lightgbm import LGBMClassifier

In [ ]:
# "A" or "B"
SWITCH = "A"

BORDERLINE = ["A_101", "A_102", "A_103", "A_104"]
FEATURES = load_variables(SWITCH)
print(f"process {SWITCH}: {len(FEATURES)} sensors")

SUB_DIR = WORKSPACE / "submissions"
SUB_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data and labels

In [ ]:
df = load_long(SWITCH, "calibration")
df_test = load_long(SWITCH, "test")

labels = pd.read_csv(INPUT_DIR / f"process {SWITCH}/calibration data/calibration_labels.csv").set_index("batch_id")["label"]
fix = labels.index.isin(BORDERLINE) & (labels != "rejected")
print("relabelled:", labels.index[fix].tolist())
labels[labels.index.isin(BORDERLINE)] = "rejected"
labels.value_counts()

## 2. Features

- `feat`: `build_features` + the extra features from `inspecting.ipynb` + batch length
- `traj`: each sensor averaged over 50 equal blocks of the batch

In [ ]:
def extra_features(d, features):
    g = d.groupby("batch_id")
    t_norm = d["t"] / g["t"].transform("max")
    phase = pd.cut(t_norm, [-0.01, 0.2, 0.8, 1.0], labels=["start", "mid", "end"])
    ph = d.groupby(["batch_id", phase], observed=True)[features].mean().unstack()
    ph.columns = [f"{f}_ph_{p}" for f, p in ph.columns]

    q = g[features].quantile([0.05, 0.95]).unstack()
    q.columns = [f"{f}_q{int(round(p * 100))}" for f, p in q.columns]

    same = d["batch_id"].eq(d["batch_id"].shift())
    rough = d[features].diff().where(same).abs().groupby(d["batch_id"]).mean()
    rough.columns = [f"{f}_rough" for f in features]

    vol = g[features].rolling(50).std().groupby(level=0).max()
    vol.columns = [f"{f}_maxvol" for f in features]

    c = g[features].corr().unstack()
    pairs = [(a, b) for i, a in enumerate(features) for b in features[i + 1:]]
    corr = c[pairs].fillna(0)
    corr.columns = [f"corr_{a}__{b}" for a, b in pairs]

    out = pd.concat([ph, q, rough, vol, corr], axis=1)
    out["length"] = g.size()
    return out


def trajectory(d, features, L=50):
    rows = {}
    for b, gg in d.sort_values(["batch_id", "t"]).groupby("batch_id"):
        a = gg[features].to_numpy(dtype=np.float64)
        rows[b] = np.stack([blk.mean(axis=0) for blk in np.array_split(a, L)]).T.ravel()
    cols = [f"{f}_tr{i:02d}" for f in features for i in range(L)]
    return pd.DataFrame.from_dict(rows, orient="index", columns=cols)


def make_views(d):
    feat = build_features(d)
    ext = extra_features(d, FEATURES)
    feat = feat.join(ext[ext.columns.difference(feat.columns)])
    return feat, trajectory(d, FEATURES)


feat, traj = make_views(df)
feat_test, traj_test = make_views(df_test)

# drop constant columns (B has some)
feat, feat_test = drop_constant(feat, feat_test[feat.columns])
traj, traj_test = drop_constant(traj, traj_test[traj.columns])

ids = labels.index.intersection(feat.index)
assert len(ids) == len(labels) == len(feat)
feat, traj = feat.loc[ids], traj.loc[ids]
traj_test = traj_test.loc[feat_test.index]

views = {"feat": feat, "feat+traj": feat.join(traj)}
views_test = {"feat": feat_test, "feat+traj": feat_test.join(traj_test)}
y = (labels.loc[ids] == "rejected").astype(int).to_numpy()  # 1 = rejected

for k in views:
    print(k, views[k].shape, views_test[k].shape)

## 3. Models

All scores: higher = rejected. PCA T² (anomaly baseline) is fit on the accepted batches of each training fold.

In [ ]:
imp = lambda: SimpleImputer(strategy="median")

def svm():
    return make_pipeline(imp(), StandardScaler(), SVC(C=1, gamma="scale", class_weight="balanced"))

MODELS = {
    "SVM rbf":        svm,
    "ExtraTrees":     lambda: make_pipeline(imp(), ExtraTreesClassifier(1000, class_weight="balanced", n_jobs=-1, random_state=0)),
    "LightGBM":       lambda: make_pipeline(imp(), LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=7,
                                  min_child_samples=5, subsample=0.8, subsample_freq=1, colsample_bytree=0.3,
                                  class_weight="balanced", verbose=-1, random_state=0)),
    "LogReg L1":      lambda: make_pipeline(imp(), StandardScaler(), LogisticRegression(C=0.3, penalty="l1",
                                  solver="liblinear", class_weight="balanced")),
}


def fit_score(make, X_tr, y_tr, X_te):
    m = make().fit(X_tr, y_tr)
    return m.decision_function(X_te) if hasattr(m, "decision_function") else m.predict_proba(X_te)[:, 1]


def fit_score_anomaly(X_tr, y_tr, X_te, n_components=15):
    m = Pipeline([("imp", imp())] + pca_monitor(n_components, "T2").steps).fit(X_tr[y_tr == 0])
    return -m.score_samples(X_te)

In [ ]:
N_SPLITS, N_REPEATS = 5, 5
splits = list(RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=0).split(ids, y))


def oof_scores(fn, X):
    # (repeats, batches); ranked within each fold so models can be averaged
    X = X.to_numpy(dtype=np.float64)
    out = np.full((N_REPEATS, len(y)), np.nan)
    for k, (tr, te) in enumerate(splits):
        s = fn(X[tr], y[tr], X[te])
        out[k // N_SPLITS, te] = rankdata(s) / len(s)
    assert not np.isnan(out).any()
    return out


oof = {}
oof["PCA T² (anomaly) | feat"] = oof_scores(fit_score_anomaly, views["feat"])
for view in views:
    for name, make in MODELS.items():
        oof[f"{name} | {view}"] = oof_scores(lambda a, b, c, make=make: fit_score(make, a, b, c), views[view])
oof["Hybrid 0.75 SVM + 0.25 T²"] = (0.75 * oof["SVM rbf | feat+traj"]
                                              + 0.25 * oof["PCA T² (anomaly) | feat"])

In [ ]:
def summarize(o):
    auc = [roc_auc_score(y, r) for r in o]
    ap = [average_precision_score(y, r) for r in o]
    return {"ROC AUC": np.mean(auc), "ROC AUC sd": np.std(auc), "PR AUC (rejected)": np.mean(ap), "PR AUC sd": np.std(ap)}

results = pd.DataFrame({k: summarize(v) for k, v in oof.items()}).T.sort_values("PR AUC (rejected)", ascending=False)
results.style.format(precision=3)

Few rejected batches (42 in A, 5 in B), so small differences are noise.

## 4. F1 with nested CV

Threshold is picked by an inner CV on the training folds, then applied to the held-out fold.

In [ ]:
def lb_metrics(y_rej, pred_rej):
    # accepted = positive; inputs use 1 = rejected
    yt, yp = 1 - np.asarray(y_rej), 1 - np.asarray(pred_rej)
    return {"accuracy": accuracy_score(yt, yp), "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0), "F1": f1_score(yt, yp, zero_division=0)}


def best_threshold(scores, y_rej):
    # F1-best cut, score > thr -> rejected; candidates are midpoints between scores
    u = np.unique(scores)
    cands = np.r_[u[0] - 1e-9, (u[:-1] + u[1:]) / 2, u[-1] + 1e-9]
    f1 = [lb_metrics(y_rej, (scores > t).astype(int))["F1"] for t in cands]
    return cands[int(np.argmax(f1))]


def inner_oof_raw(fn, X, y_, seed=0):
    out = np.full(len(y_), np.nan)
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y_):
        out[te] = fn(X[tr], y_[tr], X[te])
    return out


def nested_f1(fn, X):
    X = X.to_numpy(dtype=np.float64)
    rows, preds = [], np.zeros((N_REPEATS, len(y)), dtype=int)
    for k, (tr, te) in enumerate(splits):
        thr = best_threshold(inner_oof_raw(fn, X[tr], y[tr], seed=k), y[tr])
        preds[k // N_SPLITS, te] = (fn(X[tr], y[tr], X[te]) > thr).astype(int)
    for r in range(N_REPEATS):
        rows.append(lb_metrics(y, preds[r]))
    return pd.DataFrame(rows).agg(["mean", "std"]).T

In [ ]:
# A: SVM is best. B: most models are perfect, SVM isn't -> ExtraTrees
FINAL = {"A": ("SVM rbf", "feat+traj"), "B": ("ExtraTrees", "feat")}
FINAL_MODEL, FINAL_VIEW = FINAL[SWITCH]
final_fn = lambda a, b, c: fit_score(MODELS[FINAL_MODEL], a, b, c)

nested = {"PCA T² (anomaly) | feat": nested_f1(fit_score_anomaly, views["feat"])}
for name, view in dict.fromkeys([("SVM rbf", "feat+traj"), (FINAL_MODEL, FINAL_VIEW)]):
    nested[f"{name} | {view}"] = nested_f1(lambda a, b, c, m=MODELS[name]: fit_score(m, a, b, c), views[view])
nested["(all accepted)"] = pd.DataFrame([lb_metrics(y, np.zeros_like(y))]).agg(["mean", "std"]).T.fillna(0)

f1_table = pd.concat({k: v["mean"] for k, v in nested.items()}, axis=1).T
f1_table["F1 sd"] = [v.loc["F1", "std"] for v in nested.values()]
f1_table.style.format(precision=3)

Anomaly submission on the leaderboard (process A): F1 0.922.

## 5. Test predictions

In [ ]:
Xc = views[FINAL_VIEW].to_numpy(dtype=np.float64)
Xt = views_test[FINAL_VIEW][views[FINAL_VIEW].columns].to_numpy(dtype=np.float64)

# threshold from out-of-fold scores (5 CV runs)
oof_raw = np.concatenate([inner_oof_raw(final_fn, Xc, y, seed=s) for s in range(N_REPEATS)])
y_rep = np.tile(y, N_REPEATS)
THR = best_threshold(oof_raw, y_rep)
print(FINAL_MODEL, FINAL_VIEW, "threshold", round(THR, 4))

test_scores = pd.Series(final_fn(Xc, y, Xt), index=views_test[FINAL_VIEW].index)  # fit on all calibration
pred = np.where(test_scores > THR, "rejected", "accepted")

In [ ]:
submission = pd.DataFrame({"batch_id": test_scores.index, "label": pred}).sort_values("batch_id")
print(submission["label"].value_counts())
print("rejected share: test", round((pred == "rejected").mean(), 2), "| calibration", round(y.mean(), 2))

bins = np.linspace(min(oof_raw.min(), test_scores.min()), max(oof_raw.max(), test_scores.max()), 40)
plt.hist(oof_raw[y_rep == 0], bins=bins, alpha=0.5, density=True, label="cal accepted")
plt.hist(oof_raw[y_rep == 1], bins=bins, alpha=0.5, density=True, color="red", label="cal rejected")
plt.hist(test_scores, bins=bins, alpha=0.5, density=True, color="green", label="test")
plt.axvline(THR, color="k", ls="--", label="threshold")
plt.xlabel("score"); plt.legend(fontsize=8); plt.show()

OUT = SUB_DIR / f"submission_supervised_{SWITCH}.csv"
submission.to_csv(OUT, index=False)
print("saved", OUT)

### vs. anomaly submission

In [ ]:
prev_path = Path(f"submission_{SWITCH}.csv")
if prev_path.exists():
    prev = pd.read_csv(prev_path).set_index("batch_id")["label"]
    cmp = submission.set_index("batch_id")["label"].rename("supervised").to_frame().join(prev.rename("anomaly"))
    print("agreement:", (cmp["supervised"] == cmp["anomaly"]).mean().round(3))
    display(pd.crosstab(cmp["anomaly"], cmp["supervised"]))

else:
    print(prev_path, "not found")

## 6. Combined file (A + B)

Run once with `SWITCH = "B"`, then with `SWITCH = "A"`.

In [ ]:
template = pd.read_csv(INPUT_DIR / "BRPMM_template.csv", dtype={"batch_id": str})
parts = {p: SUB_DIR / f"submission_supervised_{p}.csv" for p in ("A", "B")}
missing = [p for p, f in parts.items() if not f.exists()]
if missing:
    print("missing:", missing)
else:
    preds = pd.concat([pd.read_csv(f, dtype={"batch_id": str}) for f in parts.values()]).set_index("batch_id")["label"]
    assert preds.index.is_unique
    assert set(preds.index) == set(template["batch_id"])
    assert set(preds.unique()) <= {"accepted", "rejected"}
    combined = template[["batch_id", "project"]].assign(label=template["batch_id"].map(preds).values)
    assert combined["label"].notna().all() and list(combined.columns) == list(template.columns)
    OUT_ALL = SUB_DIR / "submission_supervised.csv"
    combined.to_csv(OUT_ALL, index=False)
    print("saved", OUT_ALL)
    print(combined.groupby(["project", "label"]).size())